# Notebook 08b: Blockchain Deployment and Memory Persistence Proof

## Game-aware NPC: Player Based NPC Behavior Generation for Blockchain Gaming

**Author:** Ramesh Krishnan  
**Date:** February 2026  

## Objectives

This notebook deploys and validates the smart contracts created in Notebook 08a:

1. Configure deployer wallet and compile contracts
2. Run local test suite to confirm correctness
3. Deploy both contracts to Polygon Amoy testnet
4. Establish Python (web3.py) integration for reading and writing contract state
5. Seed simulated player-NPC interaction memories on-chain
6. Verify persistence by reading back stored data and validating hash integrity
7. Demonstrate NPC reading blockchain memory to reference past encounters in dialogue
8. Measure gas costs and latency per interaction

---

# Part 1: Environment Setup

The contracts and Hardhat project were created in Notebook 08a. This notebook compiles, tests, and deploys them.

In [1]:
# Cell 1: Verify project structure
# =================================
import os
from pathlib import Path

BLOCKCHAIN_DIR = Path(os.path.expanduser("~/Documents/Rkn/Learning/Masters_LJMU/Thesis/Project/game-aware-npc/blockchain"))

required_files = {
    "contracts/NPCMemory.sol": "Interaction memory contract",
    "contracts/NPCAssetSystem.sol": "NFT ownership contract",
    "scripts/deploy.js": "Deployment script",
    "hardhat.config.js": "Hardhat configuration",
    "test/NPCMemory.test.js": "NPCMemory test suite",
    "test/NPCAssetSystem.test.js": "NPCAssetSystem test suite",
}

print(f"Project directory: {BLOCKCHAIN_DIR}")
print(f"Directory exists: {BLOCKCHAIN_DIR.exists()}")
print("\nFile verification:")
print("-" * 60)

all_present = True
for filepath, description in required_files.items():
    full_path = BLOCKCHAIN_DIR / filepath
    exists = full_path.exists()
    status = "FOUND" if exists else "MISSING"
    if not exists:
        all_present = False
    print(f"  [{status}] {filepath} -- {description}")

if all_present:
    print("\nAll source files present. Ready for compilation.")
else:
    print("\nERROR: Missing files. Run Notebook 08a first.")

Project directory: /Users/rameshkrishnan/Documents/Rkn/Learning/Masters_LJMU/Thesis/Project/game-aware-npc/blockchain
Directory exists: True

File verification:
------------------------------------------------------------
  [FOUND] contracts/NPCMemory.sol -- Interaction memory contract
  [FOUND] contracts/NPCAssetSystem.sol -- NFT ownership contract
  [FOUND] scripts/deploy.js -- Deployment script
  [FOUND] hardhat.config.js -- Hardhat configuration
  [FOUND] test/NPCMemory.test.js -- NPCMemory test suite
  [FOUND] test/NPCAssetSystem.test.js -- NPCAssetSystem test suite

All source files present. Ready for compilation.


In [2]:
# Cell 2: Configure wallet and environment
# ==========================================
import getpass

DEPLOYER_ADDRESS = "0x5195C8DD882Ee654ee7794f2c91A7cc070A54D41"
AMOY_RPC_URL = "https://rpc-amoy.polygon.technology"

print("Enter the private key for the deployer wallet.")
print(f"Expected address: {DEPLOYER_ADDRESS}")
print("(Input is hidden)")
private_key = getpass.getpass("Private key: ")

clean_key = private_key.strip()
if clean_key.startswith("0x"):
    clean_key = clean_key[2:]

if len(clean_key) != 64:
    print(f"ERROR: Expected 64 hex characters, got {len(clean_key)}")
else:
    env_path = BLOCKCHAIN_DIR / ".env"
    with open(env_path, "w") as f:
        f.write(f"PRIVATE_KEY=0x{clean_key}\n")
        f.write(f"AMOY_RPC_URL={AMOY_RPC_URL}\n")
    print(f"\n.env written to: {env_path}")
    print(f"Deployer: {DEPLOYER_ADDRESS}")
    print(f"Network:  Polygon Amoy (Chain ID: 80002)")

Enter the private key for the deployer wallet.
Expected address: 0x5195C8DD882Ee654ee7794f2c91A7cc070A54D41
(Input is hidden)


Private key:  ········



.env written to: /Users/rameshkrishnan/Documents/Rkn/Learning/Masters_LJMU/Thesis/Project/game-aware-npc/blockchain/.env
Deployer: 0x5195C8DD882Ee654ee7794f2c91A7cc070A54D41
Network:  Polygon Amoy (Chain ID: 80002)


In [3]:
#Cell 2.5 align NPCMemory.sol pragma to 0.8.20
sol_path = BLOCKCHAIN_DIR / "contracts" / "NPCMemory.sol"
content = open(sol_path).read()
content = content.replace("pragma solidity ^0.8.19;", "pragma solidity ^0.8.20;")
with open(sol_path, "w") as f:
    f.write(content)
print("Fixed: NPCMemory.sol pragma updated to ^0.8.20")

Fixed: NPCMemory.sol pragma updated to ^0.8.20


In [4]:
import subprocess
# Check what's actually on disk
result = subprocess.run("head -5 contracts/NPCAssetSystem.sol", shell=True, cwd=str(BLOCKCHAIN_DIR), capture_output=True, text=True)
print("NPCAssetSystem.sol header:")
print(result.stdout)

# Check installed OpenZeppelin version
result2 = subprocess.run("cat node_modules/@openzeppelin/contracts/package.json | grep version", shell=True, cwd=str(BLOCKCHAIN_DIR), capture_output=True, text=True)
print("OpenZeppelin version:")
print(result2.stdout)

# Check what pragma OZ uses
result3 = subprocess.run("head -3 node_modules/@openzeppelin/contracts/token/ERC721/ERC721.sol", shell=True, cwd=str(BLOCKCHAIN_DIR), capture_output=True, text=True)
print("OZ ERC721 pragma:")
print(result3.stdout)

# Verbose compile
result4 = subprocess.run("npx hardhat compile --verbose 2>&1 | tail -20", shell=True, cwd=str(BLOCKCHAIN_DIR), capture_output=True, text=True)
print("Verbose output:")
print(result4.stdout)
print(result4.stderr)

NPCAssetSystem.sol header:
// SPDX-License-Identifier: MIT
// NPCAssetSystem.sol - NPC Asset Ownership and Trading Contract
// Part of: Game-aware NPC System for Blockchain Gaming
// Author: Ramesh Krishnan
// Game: "Origins of Lume: Gate of Whispers"

OpenZeppelin version:
  "version": "5.6.1",

OZ ERC721 pragma:
// SPDX-License-Identifier: MIT
// OpenZeppelin Contracts (last updated v5.6.0) (token/ERC721/ERC721.sol)


Verbose output:
2026-03-02T16:53:09.405Z hardhat:core:compilation-job File '/Users/rameshkrishnan/Documents/Rkn/Learning/Masters_LJMU/Thesis/Project/game-aware-npc/blockchain/node_modules/@openzeppelin/contracts/utils/Context.sol' added as dependency of '/Users/rameshkrishnan/Documents/Rkn/Learning/Masters_LJMU/Thesis/Project/game-aware-npc/blockchain/node_modules/@openzeppelin/contracts/access/Ownable.sol'
2026-03-02T16:53:09.405Z hardhat:core:hre Running task compile:solidity:get-compilation-job-for-file
2026-03-02T16:53:09.405Z hardhat:core:hre Running compile:solidi

In [5]:
# cell 2b Update Solidity compiler to 0.8.24 and align contract pragmas
import json

# 1. Update hardhat.config.js
config_path = BLOCKCHAIN_DIR / "hardhat.config.js"
content = open(config_path).read()
content = content.replace('"0.8.20"', '"0.8.24"')
with open(config_path, "w") as f:
    f.write(content)
print("hardhat.config.js: compiler updated to 0.8.24")

# 2. Update NPCAssetSystem.sol pragma
asset_path = BLOCKCHAIN_DIR / "contracts" / "NPCAssetSystem.sol"
content = open(asset_path).read()
content = content.replace("pragma solidity ^0.8.20;", "pragma solidity ^0.8.24;")
with open(asset_path, "w") as f:
    f.write(content)
print("NPCAssetSystem.sol: pragma updated to ^0.8.24")

# 3. Update NPCMemory.sol pragma
memory_path = BLOCKCHAIN_DIR / "contracts" / "NPCMemory.sol"
content = open(memory_path).read()
content = content.replace("pragma solidity ^0.8.19;", "pragma solidity ^0.8.24;")
content = content.replace("pragma solidity ^0.8.20;", "pragma solidity ^0.8.24;")
with open(memory_path, "w") as f:
    f.write(content)
print("NPCMemory.sol: pragma updated to ^0.8.24")

hardhat.config.js: compiler updated to 0.8.24
NPCAssetSystem.sol: pragma updated to ^0.8.24
NPCMemory.sol: pragma updated to ^0.8.24


In [6]:
# Fix: Bump to 0.8.28 (latest stable, supports mcopy/Cancun opcodes)
config_path = BLOCKCHAIN_DIR / "hardhat.config.js"
content = open(config_path).read()
content = content.replace('"0.8.24"', '"0.8.28"')
with open(config_path, "w") as f:
    f.write(content)
print("hardhat.config.js: compiler updated to 0.8.28")

# Also update contract pragmas
for sol_file in ["contracts/NPCMemory.sol", "contracts/NPCAssetSystem.sol"]:
    path = BLOCKCHAIN_DIR / sol_file
    content = open(path).read()
    content = content.replace("pragma solidity ^0.8.24;", "pragma solidity ^0.8.28;")
    with open(path, "w") as f:
        f.write(content)
    print(f"{sol_file}: pragma updated to ^0.8.28")

hardhat.config.js: compiler updated to 0.8.28
contracts/NPCMemory.sol: pragma updated to ^0.8.28
contracts/NPCAssetSystem.sol: pragma updated to ^0.8.28


In [7]:
# Fix: Add evmVersion: "cancun" to hardhat config
config_path = BLOCKCHAIN_DIR / "hardhat.config.js"
content = open(config_path).read()
content = content.replace(
    'optimizer: { enabled: true, runs: 200 }',
    'optimizer: { enabled: true, runs: 200 },\n      evmVersion: "cancun"'
)
with open(config_path, "w") as f:
    f.write(content)
print("hardhat.config.js: added evmVersion: cancun")

hardhat.config.js: added evmVersion: cancun


In [8]:
# Cell 3: Install dependencies and compile
# ==========================================
import subprocess

def run_command(cmd, cwd, description):
    print(f"\n{'=' * 60}")
    print(f"{description}")
    print(f"Command: {cmd}")
    print("=" * 60)
    result = subprocess.run(cmd, shell=True, cwd=str(cwd), capture_output=True, text=True)
    if result.stdout:
        for line in result.stdout.strip().split("\n")[-20:]:
            print(f"  {line}")
    if result.returncode != 0 and result.stderr:
        print(f"\nERROR:")
        for line in result.stderr.strip().split("\n")[-10:]:
            print(f"  {line}")
    else:
        print(f"\nCompleted successfully.")
    return result.returncode == 0

success_install = run_command("npm install", BLOCKCHAIN_DIR, "Installing npm dependencies")

if success_install:
    success_compile = run_command("npx hardhat compile", BLOCKCHAIN_DIR, "Compiling Solidity contracts")
    if success_compile:
        artifacts_dir = BLOCKCHAIN_DIR / "artifacts" / "contracts"
        print(f"\nArtifacts:")
        print(f"  NPCMemory.json:      {'FOUND' if (artifacts_dir / 'NPCMemory.sol' / 'NPCMemory.json').exists() else 'MISSING'}")
        print(f"  NPCAssetSystem.json: {'FOUND' if (artifacts_dir / 'NPCAssetSystem.sol' / 'NPCAssetSystem.json').exists() else 'MISSING'}")


Installing npm dependencies
Command: npm install
  up to date, audited 581 packages in 15s
  
  112 packages are looking for funding
    run `npm fund` for details
  
  37 vulnerabilities (21 low, 11 moderate, 5 high)
  
  To address issues that do not require attention, run:
    npm audit fix
  
  To address all issues (including breaking changes), run:
    npm audit fix --force
  
  Run `npm audit` for details.

Completed successfully.

Compiling Solidity contracts
Command: npx hardhat compile
  Nothing to compile

Completed successfully.

Artifacts:
  NPCMemory.json:      FOUND
  NPCAssetSystem.json: FOUND


In [9]:
# Cell 4: Run local test suite
# ============================
success_test = run_command("npx hardhat test", BLOCKCHAIN_DIR, "Running full test suite (local Hardhat network)")
if success_test:
    print("\nAll tests passed. Ready for testnet deployment.")
else:
    print("\nTest failures detected. Review output before deploying.")


Running full test suite (local Hardhat network)
Command: npx hardhat test
      Relationships
        ✔ Should update relationship score correctly
        ✔ Should clamp relationship at 100
        ✔ Should clamp relationship at -100
        ✔ Should track relationships per player independently
      Interaction History
        ✔ Should increment interaction count
        ✔ Should store interaction in history
        ✔ Should return correct last interaction
      Events
        ✔ Should emit InteractionRecorded event
      Player Stats
        ✔ Should return all stats in one call
      Edge Cases
        ✔ Should handle new player (no interactions)
        ✔ Should revert when getting interaction from empty history
        ✔ Should revert when accessing out-of-bounds index
  
  
    50 passing (736ms)

Completed successfully.

All tests passed. Ready for testnet deployment.


---

# Part 2: Testnet Deployment

In [10]:
# Cell 5: Check wallet balance on Amoy
# =====================================
try:
    from web3 import Web3
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "web3", "python-dotenv"], capture_output=True)
    from web3 import Web3

AMOY_RPC = "https://rpc-amoy.polygon.technology"
DEPLOYER = "0x5195C8DD882Ee654ee7794f2c91A7cc070A54D41"

w3 = Web3(Web3.HTTPProvider(AMOY_RPC))
print(f"Connected to Amoy: {w3.is_connected()}")
print(f"Chain ID: {w3.eth.chain_id}")
print(f"Latest block: {w3.eth.block_number}")

balance_wei = w3.eth.get_balance(DEPLOYER)
balance_pol = w3.from_wei(balance_wei, 'ether')
print(f"\nWallet: {DEPLOYER}")
print(f"Balance: {balance_pol:.6f} POL")

if balance_pol >= 0.2:
    print(f"\nSufficient balance for deployment.")
else:
    print(f"\nINSUFFICIENT BALANCE. Faucet: https://faucet.polygon.technology/")

Connected to Amoy: True
Chain ID: 80002
Latest block: 34671487

Wallet: 0x5195C8DD882Ee654ee7794f2c91A7cc070A54D41
Balance: 15.000000 POL

Sufficient balance for deployment.


In [11]:
# Cell 6: Deploy contracts to Amoy testnet
# ==========================================
success_deploy = run_command(
    "npx hardhat run scripts/deploy.js --network amoy",
    BLOCKCHAIN_DIR,
    "Deploying contracts to Polygon Amoy testnet"
)

if success_deploy:
    import json
    deployment_path = BLOCKCHAIN_DIR / "deployment-amoy.json"
    if deployment_path.exists():
        with open(deployment_path) as f:
            deployment = json.load(f)
        print("\n" + "=" * 60)
        print("DEPLOYMENT SUMMARY")
        print("=" * 60)
        print(f"Network:     {deployment.get('network', 'amoy')}")
        print(f"Chain ID:    {deployment.get('chainId', 80002)}")
        print(f"Deployer:    {deployment.get('deployer', 'unknown')}")
        print(f"Deployed at: {deployment.get('deployedAt', 'unknown')}")
        contracts = deployment.get('contracts', {})
        print(f"\nContract Addresses:")
        for name, addr in contracts.items():
            a = addr.get('address', addr) if isinstance(addr, dict) else addr
            print(f"  {name}: {a}")
        explorer = deployment.get('explorer', {})
        if isinstance(explorer, dict):
            print(f"\nPolygonScan Links:")
            for name, url in explorer.items():
                print(f"  {name}: {url}")
else:
    print("\nDeployment failed. Check wallet balance and network connectivity.")


Deploying contracts to Polygon Amoy testnet
Command: npx hardhat run scripts/deploy.js --network amoy
  
  [Init] Creating Whisper NPC on-chain...
    ✅ Whisper created with NPC ID: 1
    ✅ Whisper funded with initial balance
  
  ✅ Deployment info saved to: /Users/rameshkrishnan/Documents/Rkn/Learning/Masters_LJMU/Thesis/Project/game-aware-npc/blockchain/deployment-amoy.json
  
  DEPLOYMENT COMPLETE
  
  Contract Addresses:
    NPCMemory:      0xaD523d0b1CB1772917eC16BE585CBb04eD7b079a
    NPCAssetSystem: 0x3cF49695BE6A1b51e46E91761B8C2c7EF809694f
  
  View on PolygonScan:
     https://amoy.polygonscan.com/address/0xaD523d0b1CB1772917eC16BE585CBb04eD7b079a
     https://amoy.polygonscan.com/address/0x3cF49695BE6A1b51e46E91761B8C2c7EF809694f
  

Completed successfully.

DEPLOYMENT SUMMARY
Network:     amoy
Chain ID:    80002
Deployer:    0x5195C8DD882Ee654ee7794f2c91A7cc070A54D41
Deployed at: 2026-03-02T16:54:38.487Z

Contract Addresses:
  NPCMemory: 0xaD523d0b1CB1772917eC16BE585CBb04e

---

# Part 3: Python Integration via web3.py

In [12]:
# Cell 7: Load ABIs and create contract objects
# ===============================================
import json
from web3 import Web3
from pathlib import Path

deployment_path = BLOCKCHAIN_DIR / "deployment-amoy.json"
with open(deployment_path) as f:
    deployment = json.load(f)

contracts_info = deployment.get("contracts", {})

def get_address(d):
    return d.get("address", d) if isinstance(d, dict) else d

MEMORY_ADDRESS = Web3.to_checksum_address(get_address(contracts_info["NPCMemory"]))
ASSET_ADDRESS = Web3.to_checksum_address(get_address(contracts_info["NPCAssetSystem"]))

artifacts_dir = BLOCKCHAIN_DIR / "artifacts" / "contracts"
with open(artifacts_dir / "NPCMemory.sol" / "NPCMemory.json") as f:
    memory_artifact = json.load(f)
with open(artifacts_dir / "NPCAssetSystem.sol" / "NPCAssetSystem.json") as f:
    asset_artifact = json.load(f)

w3 = Web3(Web3.HTTPProvider("https://rpc-amoy.polygon.technology"))
assert w3.is_connected(), "Failed to connect to Amoy RPC"

npc_memory = w3.eth.contract(address=MEMORY_ADDRESS, abi=memory_artifact["abi"])
npc_assets = w3.eth.contract(address=ASSET_ADDRESS, abi=asset_artifact["abi"])

from dotenv import load_dotenv
import os
load_dotenv(BLOCKCHAIN_DIR / ".env")
PRIVATE_KEY = os.getenv("PRIVATE_KEY")
DEPLOYER = w3.eth.account.from_key(PRIVATE_KEY).address

print("Contract Connection Summary")
print("=" * 60)
print(f"Network:            Polygon Amoy (Chain ID: {w3.eth.chain_id})")
print(f"Deployer/Owner:     {DEPLOYER}")
print(f"NPCMemory:          {MEMORY_ADDRESS}")
print(f"NPCAssetSystem:     {ASSET_ADDRESS}")
print(f"\nOwnership verification:")
print(f"  NPCMemory owner:      {npc_memory.functions.owner().call()}")
print(f"  NPCAssetSystem owner: {npc_assets.functions.owner().call()}")
print(f"  Matches deployer:     {npc_memory.functions.owner().call() == DEPLOYER}")

total_npcs = npc_assets.functions.getTotalNPCs().call()
print(f"\nWhisper NPC:")
print(f"  Total NPCs on-chain: {total_npcs}")
if total_npcs > 0:
    whisper = npc_assets.functions.getNPC(1).call()
    print(f"  NPC #1 name:   {whisper[1]}")
    print(f"  NPC #1 type:   {whisper[2]}")
    print(f"  NPC #1 active: {whisper[3]}")

Contract Connection Summary
Network:            Polygon Amoy (Chain ID: 80002)
Deployer/Owner:     0x5195C8DD882Ee654ee7794f2c91A7cc070A54D41
NPCMemory:          0xaD523d0b1CB1772917eC16BE585CBb04eD7b079a
NPCAssetSystem:     0x3cF49695BE6A1b51e46E91761B8C2c7EF809694f

Ownership verification:
  NPCMemory owner:      0x5195C8DD882Ee654ee7794f2c91A7cc070A54D41
  NPCAssetSystem owner: 0x5195C8DD882Ee654ee7794f2c91A7cc070A54D41
  Matches deployer:     True

Whisper NPC:
  Total NPCs on-chain: 1
  NPC #1 name:   Whisper
  NPC #1 type:   merchant
  NPC #1 active: True


In [13]:
# Cell 8: Transaction helper
# ===========================
import time

def send_transaction(tx_func):
    tx = tx_func.build_transaction({
        'from': DEPLOYER,
        'nonce': w3.eth.get_transaction_count(DEPLOYER),
        'gas': 300000,
        'gasPrice': w3.eth.gas_price,
        'chainId': 80002,
    })
    signed = w3.eth.account.sign_transaction(tx, PRIVATE_KEY)
    tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=120)
    return receipt

print("Transaction helper loaded.")
print(f"Current nonce: {w3.eth.get_transaction_count(DEPLOYER)}")

Transaction helper loaded.
Current nonce: 4


---

# Part 4: Seed Interaction Memories On-Chain

8 simulated interactions across 6 player archetypes (from Notebook 01 clustering).

In [14]:
# Cell 9: Define simulated interaction scenarios
# ================================================
SIMULATED_PLAYERS = {
    "trophy_hunter": "0x1000000000000000000000000000000000000001",
    "engaged_beginner": "0x2000000000000000000000000000000000000002",
    "achievement_hunter": "0x3000000000000000000000000000000000000003",
    "casual_veteran": "0x4000000000000000000000000000000000000004",
    "weekend_warrior": "0x5000000000000000000000000000000000000005",
    "spender": "0x6000000000000000000000000000000000000006",
}

SEED_INTERACTIONS = [
    {"archetype": "trophy_hunter", "summary": "Player asked about rare scrolls. Whisper offered Gate Scroll at 50 points. Player purchased after brief negotiation.", "delta": 10},
    {"archetype": "trophy_hunter", "summary": "Player returned asking for discount on second scroll. Whisper denied -- relationship not high enough. Player accepted.", "delta": 3},
    {"archetype": "engaged_beginner", "summary": "New player confused about game mechanics. Whisper provided hints about curses and golden gates. No purchase.", "delta": 8},
    {"archetype": "achievement_hunter", "summary": "Player asked about NFTs. Whisper explained Merchant Favor benefits. Player purchased for 15 POL.", "delta": 15},
    {"archetype": "casual_veteran", "summary": "Returning player greeted Whisper. Brief conversation about game progress. No purchase.", "delta": 5},
    {"archetype": "weekend_warrior", "summary": "Player attempted to haggle aggressively on hint prices. Whisper held firm. No purchase.", "delta": -3},
    {"archetype": "spender", "summary": "High-value player purchased Shadow Blessing (25 POL) and two Gate Scrolls. Whisper applied loyalty discount.", "delta": 20},
    {"archetype": "spender", "summary": "Player inquired about exclusive items. Whisper mentioned rarer items at higher trust levels.", "delta": 12},
    {"archetype": "engaged_beginner", "summary": "Player requested a loan of 200 points. Whisper approved based on relationship score. Debt recorded: 200 pts.", "delta": 5},
    {"archetype": "engaged_beginner", "summary": "Player returned with outstanding debt of 200 pts. Whisper requested repayment before new transactions. Player repaid in full.", "delta": 10},
]

print(f"Prepared {len(SEED_INTERACTIONS)} interactions across {len(SIMULATED_PLAYERS)} archetypes")
for i, ix in enumerate(SEED_INTERACTIONS):
    sign = "+" if ix['delta'] > 0 else ""
    print(f"  [{i+1}] {ix['archetype']:20s} | delta: {sign}{ix['delta']:3d} | {ix['summary'][:55]}...")

Prepared 10 interactions across 6 archetypes
  [1] trophy_hunter        | delta: + 10 | Player asked about rare scrolls. Whisper offered Gate S...
  [2] trophy_hunter        | delta: +  3 | Player returned asking for discount on second scroll. W...
  [3] engaged_beginner     | delta: +  8 | New player confused about game mechanics. Whisper provi...
  [4] achievement_hunter   | delta: + 15 | Player asked about NFTs. Whisper explained Merchant Fav...
  [5] casual_veteran       | delta: +  5 | Returning player greeted Whisper. Brief conversation ab...
  [6] weekend_warrior      | delta:  -3 | Player attempted to haggle aggressively on hint prices....
  [7] spender              | delta: + 20 | High-value player purchased Shadow Blessing (25 POL) an...
  [8] spender              | delta: + 12 | Player inquired about exclusive items. Whisper mentione...
  [9] engaged_beginner     | delta: +  5 | Player requested a loan of 200 points. Whisper approved...
  [10] engaged_beginner     | delta: +

In [15]:
# Cell 10: Record interactions on-chain
# =======================================
results = []
print("Recording interactions on Polygon Amoy...")
print("=" * 70)

for i, interaction in enumerate(SEED_INTERACTIONS):
    archetype = interaction["archetype"]
    player_addr = Web3.to_checksum_address(SIMULATED_PLAYERS[archetype])
    interaction_hash = Web3.keccak(text=interaction["summary"])
    delta = interaction["delta"]
    
    print(f"\n[{i+1}/{len(SEED_INTERACTIONS)}] {archetype} (delta: {delta:+d})")
    print(f"  Player: {player_addr}")
    
    start_time = time.time()
    try:
        receipt = send_transaction(
            npc_memory.functions.recordInteraction(player_addr, interaction_hash, delta)
        )
        elapsed = time.time() - start_time
        results.append({
            'index': i, 'archetype': archetype, 'player': player_addr,
            'delta': delta, 'interaction_hash': interaction_hash.hex(),
            'tx_hash': receipt['transactionHash'].hex(),
            'gas_used': receipt['gasUsed'], 'elapsed_seconds': elapsed,
            'block_number': receipt['blockNumber'],
            'status': 'success' if receipt['status'] == 1 else 'failed',
        })
        print(f"  TX: {receipt['transactionHash'].hex()[:32]}...")
        print(f"  Gas: {receipt['gasUsed']:,} | Time: {elapsed:.2f}s | Block: {receipt['blockNumber']}")
    except Exception as e:
        print(f"  ERROR: {e}")
        results.append({'index': i, 'archetype': archetype, 'status': 'error', 'error': str(e)})

successful = [r for r in results if r['status'] == 'success']
print(f"\n{'=' * 70}")
print(f"Seeding complete: {len(successful)}/{len(SEED_INTERACTIONS)} recorded.")

Recording interactions on Polygon Amoy...

[1/10] trophy_hunter (delta: +10)
  Player: 0x1000000000000000000000000000000000000001
  TX: 6a6007fb372ba11443349abc2249f184...
  Gas: 204,693 | Time: 2.80s | Block: 34671698

[2/10] trophy_hunter (delta: +3)
  Player: 0x1000000000000000000000000000000000000001
  TX: 6728f86b05955b005a9666123b5b0e29...
  Gas: 136,293 | Time: 4.12s | Block: 34671700

[3/10] engaged_beginner (delta: +8)
  Player: 0x2000000000000000000000000000000000000002
  TX: a88c0bd6c889c3d28910b2e18a6cf8cf...
  Gas: 187,593 | Time: 3.89s | Block: 34671702

[4/10] achievement_hunter (delta: +15)
  Player: 0x3000000000000000000000000000000000000003
  TX: 9721f98a29dea10401d9531f520ac036...
  Gas: 187,593 | Time: 3.95s | Block: 34671704

[5/10] casual_veteran (delta: +5)
  Player: 0x4000000000000000000000000000000000000004
  TX: c8b5888b540e7de8bac4dfaf5497d6e0...
  Gas: 187,593 | Time: 4.21s | Block: 34671706

[6/10] weekend_warrior (delta: -3)
  Player: 0x5000000000000000000

---

# Part 5: Persistence Verification

In [16]:
# Cell 11: Read back and verify all on-chain data
# =================================================
EXPLORER_BASE = "https://amoy.polygonscan.com"

print("BLOCKCHAIN MEMORY VERIFICATION")
print("=" * 70)

expected = {}
for ix in SEED_INTERACTIONS:
    addr = Web3.to_checksum_address(SIMULATED_PLAYERS[ix['archetype']])
    if addr not in expected:
        expected[addr] = {'count': 0, 'relationship': 0, 'archetype': ix['archetype']}
    expected[addr]['count'] += 1
    expected[addr]['relationship'] += ix['delta']
    expected[addr]['relationship'] = max(-100, min(100, expected[addr]['relationship']))

all_verified = True
for addr, exp in expected.items():
    on_chain_rel = npc_memory.functions.getRelationship(addr).call()
    on_chain_count = npc_memory.functions.getInteractionCount(addr).call()
    rel_ok = on_chain_rel == exp['relationship']
    count_ok = on_chain_count == exp['count']
    status = "VERIFIED" if (rel_ok and count_ok) else "MISMATCH"
    if not (rel_ok and count_ok):
        all_verified = False
    print(f"\n  {exp['archetype']} ({addr[:10]}...{addr[-4:]})")
    print(f"    Relationship: on-chain={on_chain_rel:+d}, expected={exp['relationship']:+d} [{status}]")
    print(f"    Interactions: on-chain={on_chain_count}, expected={exp['count']} [{status}]")

print(f"\n{'=' * 70}")
print("INTERACTION HASH VERIFICATION")
print("=" * 70)

total_on_chain = npc_memory.functions.getTotalInteractions().call()
print(f"Total interactions on-chain: {total_on_chain}")

hash_mismatches = 0
for i in range(total_on_chain):
    player, timestamp, stored_hash, delta = npc_memory.functions.getInteractionByIndex(i).call()
    if i < len(SEED_INTERACTIONS):
        expected_hash = Web3.keccak(text=SEED_INTERACTIONS[i]['summary']).hex()
        stored_hex = stored_hash.hex() if isinstance(stored_hash, bytes) else stored_hash
        match = stored_hex == expected_hash
        if not match:
            hash_mismatches += 1
        status = "MATCH" if match else "MISMATCH"
    else:
        status = "EXTRA"
    print(f"  [{i}] player={player[:10]}... delta={delta:+d} hash={stored_hash.hex()[:16]}... [{status}]")

print(f"\n{'=' * 70}")
if all_verified and hash_mismatches == 0:
    print(f"RESULT: All {total_on_chain} interactions verified. Memory persistence: 100% accuracy.")
else:
    print(f"RESULT: Verification issues. {hash_mismatches} hash mismatches.")

BLOCKCHAIN MEMORY VERIFICATION

  trophy_hunter (0x10000000...0001)
    Relationship: on-chain=+13, expected=+13 [VERIFIED]
    Interactions: on-chain=2, expected=2 [VERIFIED]

  engaged_beginner (0x20000000...0002)
    Relationship: on-chain=+23, expected=+23 [VERIFIED]
    Interactions: on-chain=3, expected=3 [VERIFIED]

  achievement_hunter (0x30000000...0003)
    Relationship: on-chain=+15, expected=+15 [VERIFIED]
    Interactions: on-chain=1, expected=1 [VERIFIED]

  casual_veteran (0x40000000...0004)
    Relationship: on-chain=+5, expected=+5 [VERIFIED]
    Interactions: on-chain=1, expected=1 [VERIFIED]

  weekend_warrior (0x50000000...0005)
    Relationship: on-chain=-3, expected=-3 [VERIFIED]
    Interactions: on-chain=1, expected=1 [VERIFIED]

  spender (0x60000000...0006)
    Relationship: on-chain=+32, expected=+32 [VERIFIED]
    Interactions: on-chain=2, expected=2 [VERIFIED]

INTERACTION HASH VERIFICATION
Total interactions on-chain: 10
  [0] player=0x10000000... delta=+1

In [17]:
# Cell 12: PolygonScan verification links
# =========================================
print("INDEPENDENT VERIFICATION LINKS")
print("=" * 70)
print(f"Contract addresses:")
print(f"  NPCMemory:      {EXPLORER_BASE}/address/{MEMORY_ADDRESS}")
print(f"  NPCAssetSystem: {EXPLORER_BASE}/address/{ASSET_ADDRESS}")
print(f"\nTransaction links:")
for r in results:
    if r['status'] == 'success':
        print(f"  [{r['archetype']:20s}] {EXPLORER_BASE}/tx/{r['tx_hash']}")

INDEPENDENT VERIFICATION LINKS
Contract addresses:
  NPCMemory:      https://amoy.polygonscan.com/address/0xaD523d0b1CB1772917eC16BE585CBb04eD7b079a
  NPCAssetSystem: https://amoy.polygonscan.com/address/0x3cF49695BE6A1b51e46E91761B8C2c7EF809694f

Transaction links:
  [trophy_hunter       ] https://amoy.polygonscan.com/tx/6a6007fb372ba11443349abc2249f184cdd7a4419439213d2ba03537de1b35f8
  [trophy_hunter       ] https://amoy.polygonscan.com/tx/6728f86b05955b005a9666123b5b0e2932e767ac27c8ba2776d360233fd5d6f3
  [engaged_beginner    ] https://amoy.polygonscan.com/tx/a88c0bd6c889c3d28910b2e18a6cf8cfdba9019cd5bfc9bbe069c90367ce37ab
  [achievement_hunter  ] https://amoy.polygonscan.com/tx/9721f98a29dea10401d9531f520ac036c2b9674fb4f376fb92b447e237dc36bb
  [casual_veteran      ] https://amoy.polygonscan.com/tx/c8b5888b540e7de8bac4dfaf5497d6e02da6f80e946fb01159907e74d0354231
  [weekend_warrior     ] https://amoy.polygonscan.com/tx/7801f187a4b4b23e8c090b645db04fa90eeb77809196a5a20cd93435bb0d6ba4
 

---

# Part 6: Chat Demo -- NPC Reads Blockchain Memory

In [18]:
# Cell 13: Build system prompt from blockchain memory
# ====================================================
from datetime import datetime

def build_memory_context(player_address):
    addr = Web3.to_checksum_address(player_address)
    relationship = npc_memory.functions.getRelationship(addr).call()
    count = npc_memory.functions.getInteractionCount(addr).call()
    last_time = npc_memory.functions.getLastInteractionTime(addr).call()
    
    if count == 0:
        return "[BLOCKCHAIN MEMORY] New player. No prior interactions."
    
    if relationship >= 50: tier = "Trusted Ally"
    elif relationship >= 20: tier = "Returning Customer"
    elif relationship >= 0: tier = "Acquaintance"
    else: tier = "Wary Stranger"
    
    last_seen = datetime.utcfromtimestamp(last_time).strftime('%Y-%m-%d %H:%M UTC') if last_time > 0 else "unknown"
    
    return (
        f"[BLOCKCHAIN MEMORY]\n"
        f"Player wallet: {addr[:10]}...{addr[-4:]}\n"
        f"Relationship score: {relationship}/100 (Tier: {tier})\n"
        f"Total past interactions: {count}\n"
        f"Last seen: {last_seen}\n"
        f"On-chain verification: {EXPLORER_BASE}/address/{MEMORY_ADDRESS}"
    )

print("BLOCKCHAIN MEMORY CONTEXT GENERATION")
print("=" * 70)
for archetype, addr in SIMULATED_PLAYERS.items():
    context = build_memory_context(addr)
    print(f"\n--- {archetype} ---")
    print(context)

BLOCKCHAIN MEMORY CONTEXT GENERATION


/var/folders/41/w2wftv1j3_g46ghtyl9ddmvh0000gn/T/ipykernel_6983/1005850602.py:19: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  last_seen = datetime.utcfromtimestamp(last_time).strftime('%Y-%m-%d %H:%M UTC') if last_time > 0 else "unknown"



--- trophy_hunter ---
[BLOCKCHAIN MEMORY]
Player wallet: 0x10000000...0001
Relationship score: 13/100 (Tier: Acquaintance)
Total past interactions: 2
Last seen: 2026-03-02 17:01 UTC
On-chain verification: https://amoy.polygonscan.com/address/0xaD523d0b1CB1772917eC16BE585CBb04eD7b079a

--- engaged_beginner ---
[BLOCKCHAIN MEMORY]
Player wallet: 0x20000000...0002
Relationship score: 23/100 (Tier: Returning Customer)
Total past interactions: 3
Last seen: 2026-03-02 17:01 UTC
On-chain verification: https://amoy.polygonscan.com/address/0xaD523d0b1CB1772917eC16BE585CBb04eD7b079a

--- achievement_hunter ---
[BLOCKCHAIN MEMORY]
Player wallet: 0x30000000...0003
Relationship score: 15/100 (Tier: Acquaintance)
Total past interactions: 1
Last seen: 2026-03-02 17:01 UTC
On-chain verification: https://amoy.polygonscan.com/address/0xaD523d0b1CB1772917eC16BE585CBb04eD7b079a

--- casual_veteran ---
[BLOCKCHAIN MEMORY]
Player wallet: 0x40000000...0004
Relationship score: 5/100 (Tier: Acquaintance)
Tota

In [19]:
# Cell 14: Simulated chat with blockchain memory
# =================================================
def simulated_whisper_response(archetype, message, memory_context):
    relationship = 0
    count = 0
    for line in memory_context.split("\n"):
        if "Relationship score" in line:
            try: relationship = int(line.split(":")[1].split("/")[0].strip())
            except: pass
        if "Total past interactions" in line:
            try: count = int(line.split(":")[1].strip())
            except: pass
    
    responses = {
        "trophy_hunter": f"Ah, back again... That makes {count} times now. With trust at {relationship}, I might have something special. What brings you?",
        "engaged_beginner": f"Welcome back. You cleared your debt last time -- that earned you trust. Sitting at {relationship} now. What do you need today?",
        "achievement_hunter": f"The collector returns. With {count} interactions and trust at {relationship}, you are becoming valued. Shall we discuss my wares?",
        "casual_veteran": f"You return at your own pace. {count} visits, trust at {relationship}. What can Whisper do for you?",
        "weekend_warrior": f"Hmm. Last time did not end well. Trust sits at {relationship}. Perhaps this time we find common ground?",
        "spender": f"My most generous visitor. {count} encounters, trust at {relationship}. I may have found something worth your attention...",
    }
    return responses.get(archetype, "The shadows stir... who approaches?")

demo_messages = {
    "trophy_hunter": "Hey Whisper, got any scrolls today?",
    "engaged_beginner": "Hi, I am back. Still confused about the golden gates.",
    "achievement_hunter": "What items do you have? I want to collect everything.",
    "casual_veteran": "Just checking in. Anything new?",
    "weekend_warrior": "Can I get a better deal this time?",
    "spender": "Show me your best items.",
}

print("SIMULATED CHAT: Blockchain Memory-Aware NPC")
print("=" * 70)
for archetype, message in demo_messages.items():
    memory = build_memory_context(SIMULATED_PLAYERS[archetype])
    response = simulated_whisper_response(archetype, message, memory)
    print(f"\n{'=' * 70}")
    print(f"Player ({archetype}): {message}")
    print(f"Whisper: {response}")
    print(f"  [Memory source: {EXPLORER_BASE}/address/{MEMORY_ADDRESS}]")

SIMULATED CHAT: Blockchain Memory-Aware NPC


/var/folders/41/w2wftv1j3_g46ghtyl9ddmvh0000gn/T/ipykernel_6983/1005850602.py:19: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  last_seen = datetime.utcfromtimestamp(last_time).strftime('%Y-%m-%d %H:%M UTC') if last_time > 0 else "unknown"



Player (trophy_hunter): Hey Whisper, got any scrolls today?
Whisper: Ah, back again... That makes 2 times now. With trust at 13, I might have something special. What brings you?
  [Memory source: https://amoy.polygonscan.com/address/0xaD523d0b1CB1772917eC16BE585CBb04eD7b079a]

Player (engaged_beginner): Hi, I am back. Still confused about the golden gates.
Whisper: Welcome back. You cleared your debt last time -- that earned you trust. Sitting at 23 now. What do you need today?
  [Memory source: https://amoy.polygonscan.com/address/0xaD523d0b1CB1772917eC16BE585CBb04eD7b079a]

Player (achievement_hunter): What items do you have? I want to collect everything.
Whisper: The collector returns. With 1 interactions and trust at 15, you are becoming valued. Shall we discuss my wares?
  [Memory source: https://amoy.polygonscan.com/address/0xaD523d0b1CB1772917eC16BE585CBb04eD7b079a]

Player (casual_veteran): Just checking in. Anything new?
Whisper: You return at your own pace. 1 visits, trust a

In [20]:
# Cell 15: Record new interaction (write-read-use-write cycle)
# ==============================================================
new_summary = "Spender returned asking for exclusive items. Whisper referenced past purchase from blockchain memory."
new_hash = Web3.keccak(text=new_summary)
spender_addr = Web3.to_checksum_address(SIMULATED_PLAYERS["spender"])

print("Recording new interaction from chat demo...")
receipt = send_transaction(
    npc_memory.functions.recordInteraction(spender_addr, new_hash, 8)
)

print(f"  TX: {receipt['transactionHash'].hex()}")
print(f"  Gas: {receipt['gasUsed']:,} | Status: {'confirmed' if receipt['status'] == 1 else 'FAILED'}")

updated_rel = npc_memory.functions.getRelationship(spender_addr).call()
updated_count = npc_memory.functions.getInteractionCount(spender_addr).call()
print(f"\nUpdated spender stats:")
print(f"  Relationship: {updated_rel} (was 32, added +8, expected 40)")
print(f"  Interactions: {updated_count} (was 2, expected 3)")
print(f"  Verify: {EXPLORER_BASE}/tx/{receipt['transactionHash'].hex()}")

Recording new interaction from chat demo...
  TX: 7fa3b9f5938417369c2c7a8152e5326384333ea071caa9531d66e88ee7c3de1a
  Gas: 136,293 | Status: confirmed

Updated spender stats:
  Relationship: 40 (was 32, added +8, expected 40)
  Interactions: 3 (was 2, expected 3)
  Verify: https://amoy.polygonscan.com/tx/7fa3b9f5938417369c2c7a8152e5326384333ea071caa9531d66e88ee7c3de1a


---

# Part 7: Gas Cost and Performance Analysis

In [21]:
# Cell 16: Gas cost and latency analysis
# ========================================
import statistics

successful_results = [r for r in results if r['status'] == 'success']

if successful_results:
    gas_values = [r['gas_used'] for r in successful_results]
    time_values = [r['elapsed_seconds'] for r in successful_results]
    
    avg_gas = statistics.mean(gas_values)
    GAS_PRICE_GWEI = 30
    POL_PRICE_USD = 0.50
    avg_cost_pol = avg_gas * GAS_PRICE_GWEI * 1e-9
    avg_cost_usd = avg_cost_pol * POL_PRICE_USD
    
    print("GAS COST ANALYSIS")
    print("=" * 60)
    print(f"Transactions: {len(successful_results)}")
    print(f"Gas/interaction: avg={avg_gas:,.0f}, min={min(gas_values):,}, max={max(gas_values):,}")
    print(f"Cost/interaction: {avg_cost_pol:.6f} POL (~${avg_cost_usd:.4f} USD)")
    print(f"Target (H1.2): < $0.01")
    print(f"Result: {'PASS' if avg_cost_usd < 0.01 else 'FAIL'} -- ${avg_cost_usd:.4f} < $0.01")
    print(f"\nProjected study cost: 15 users x 20 interactions = ${300 * avg_cost_usd:.2f} USD")
    
    avg_time = statistics.mean(time_values)
    print(f"\nTRANSACTION LATENCY")
    print("=" * 60)
    print(f"Confirmation time: avg={avg_time:.2f}s, min={min(time_values):.2f}s, max={max(time_values):.2f}s")
    print(f"Note: Blockchain writes are asynchronous in production.")
    print(f"The player does not wait for confirmation.")

GAS COST ANALYSIS
Transactions: 10
Gas/interaction: avg=168,820, min=136,293, max=204,693
Cost/interaction: 0.005065 POL (~$0.0025 USD)
Target (H1.2): < $0.01
Result: PASS -- $0.0025 < $0.01

Projected study cost: 15 users x 20 interactions = $0.76 USD

TRANSACTION LATENCY
Confirmation time: avg=3.90s, min=2.80s, max=4.21s
Note: Blockchain writes are asynchronous in production.
The player does not wait for confirmation.


In [22]:
# Cell 17: Transaction log
# =========================
print("TRANSACTION LOG")
print("=" * 100)
print(f"{'#':>3} {'Archetype':>20} {'Delta':>6} {'Gas':>10} {'Time(s)':>8} {'Block':>10} {'TX Hash':>20}")
print("-" * 100)
for r in successful_results:
    print(f"{r['index']+1:>3} {r['archetype']:>20} {r['delta']:>+6d} {r['gas_used']:>10,} {r['elapsed_seconds']:>8.2f} {r['block_number']:>10} {r['tx_hash'][:16]:>20}")
print("-" * 100)

TRANSACTION LOG
  #            Archetype  Delta        Gas  Time(s)      Block              TX Hash
----------------------------------------------------------------------------------------------------
  1        trophy_hunter    +10    204,693     2.80   34671698     6a6007fb372ba114
  2        trophy_hunter     +3    136,293     4.12   34671700     6728f86b05955b00
  3     engaged_beginner     +8    187,593     3.89   34671702     a88c0bd6c889c3d2
  4   achievement_hunter    +15    187,593     3.95   34671704     9721f98a29dea104
  5       casual_veteran     +5    187,593     4.21   34671706     c8b5888b540e7de8
  6      weekend_warrior     -3    187,965     3.84   34671708     7801f187a4b4b23e
  7              spender    +20    187,593     4.16   34671710     8ee465548cf215df
  8              spender    +12    136,293     3.96   34671712     feda2169fb9ce8f7
  9     engaged_beginner     +5    136,293     4.05   34671714     b3199c2bb63edbeb
 10     engaged_beginner    +10    136,293 

In [23]:
# Cell 18: Export results
# ========================
import json
from datetime import datetime

output = {
    "notebook": "08b_Blockchain_Deployment_and_Memory_Proof",
    "execution_date": datetime.utcnow().isoformat() + "Z",
    "network": "Polygon Amoy Testnet (Chain ID: 80002)",
    "deployer_wallet": DEPLOYER,
    "contracts": {"NPCMemory": MEMORY_ADDRESS, "NPCAssetSystem": ASSET_ADDRESS},
    "explorer_links": {
        "NPCMemory": f"{EXPLORER_BASE}/address/{MEMORY_ADDRESS}",
        "NPCAssetSystem": f"{EXPLORER_BASE}/address/{ASSET_ADDRESS}",
    },
    "test_suite": "All local tests passed",
    "seeded_interactions": len(SEED_INTERACTIONS),
    "verification_result": "100% accuracy" if all_verified else "mismatches detected",
    "gas_analysis": {
        "avg_gas_per_interaction": round(statistics.mean(gas_values)),
        "avg_cost_usd": round(avg_cost_usd, 6),
        "h1_2_target_met": avg_cost_usd < 0.01,
    },
    "transactions": successful_results,
}

output_path = BLOCKCHAIN_DIR / "blockchain_memory_proof_results.json"
with open(output_path, "w") as f:
    json.dump(output, f, indent=2, default=str)

print(f"Results exported to: {output_path}")
print(f"\nKey findings:")
print(f"  Memory persistence:   {output['verification_result']}")
print(f"  Avg cost/interaction: ${output['gas_analysis']['avg_cost_usd']}")
print(f"  H1.2 target met:     {output['gas_analysis']['h1_2_target_met']}")

Results exported to: /Users/rameshkrishnan/Documents/Rkn/Learning/Masters_LJMU/Thesis/Project/game-aware-npc/blockchain/blockchain_memory_proof_results.json

Key findings:
  Memory persistence:   100% accuracy
  Avg cost/interaction: $0.002532
  H1.2 target met:     True


/var/folders/41/w2wftv1j3_g46ghtyl9ddmvh0000gn/T/ipykernel_6983/3965620356.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "execution_date": datetime.utcnow().isoformat() + "Z",


---

# Summary

**Deployment:** Both contracts deployed to Polygon Amoy. Whisper NPC created on-chain (ID 1).

**Verification:** All 8 seeded interactions verified with 100% hash accuracy. Independent verification available via PolygonScan links.

**Chat Demo:** Whisper reads blockchain memory to reference past encounters, demonstrating Game-aware NPC capability.

**Performance:** Gas costs approximately $0.002 per interaction, well under $0.01 target (H1.2). Blockchain writes are asynchronous in production.

### Evidence Summary

| Claim | Evidence | Location |
|-------|----------|----------|
| H1.2: Gas < $0.01 | Transaction receipts | Part 7, Cell 16 |
| H3.2: Memory persistence | 100% hash verification | Part 5, Cell 11 |
| Objective 2: NPC autonomy | NPC reads blockchain state | Part 6, Cells 13-14 |
| Write-read-use-write cycle | New interaction after chat | Part 6, Cell 15 |

### Output Files

| File | Content |
|------|---------|
| deployment-amoy.json | Contract addresses and deployment metadata |
| blockchain_memory_proof_results.json | Verification data, gas analysis, transaction log |